# Notebook 07: Integration — All 6 CCA Patterns in Action

This notebook demonstrates all 6 CCA architectural patterns in a single customer interaction.
One scenario — C003 Carol Martinez, \$600 refund with PII in message — exercises every pattern.

**The 6 patterns:**

| # | Pattern | Correct Approach | Anti-Pattern |
|---|---------|-----------------|--------------|
| 1 | Escalation | Deterministic callback rules | LLM confidence scores |
| 2 | Compliance | Programmatic PII redaction | Prompt-only rules |
| 3 | Tool Design | 5 focused tools | 15+ Swiss Army tools |
| 4 | Context | Structured summaries | Raw transcripts |
| 5 | Cost Optimization | Prompt caching | Batch API for live support |
| 6 | Handoffs | Structured EscalationRecord | Raw conversation dump |

## Setup

This notebook demonstrates all 6 CCA patterns in a single customer interaction.

In [ ]:
import json
import sys
from pathlib import Path

# Add project root so notebooks.helpers and customer_service are importable
sys.path.insert(0, str(Path(".").resolve()))

import anthropic
from helpers import estimate_cost

from customer_service.agent import (
    TOKEN_BUDGET,
    ContextSummary,
    build_callbacks,
    get_system_prompt,
    get_system_prompt_with_caching,
    run_agent_loop,
)
from customer_service.anti_patterns import format_raw_handoff
from customer_service.data.customers import CUSTOMERS
from customer_service.services.audit_log import AuditLog
from customer_service.services.container import ServiceContainer
from customer_service.services.customer_db import CustomerDatabase
from customer_service.services.escalation_queue import EscalationQueue
from customer_service.services.financial_system import FinancialSystem
from customer_service.services.policy_engine import PolicyEngine

In [ ]:
def make_services() -> ServiceContainer:
    """Create a fresh ServiceContainer with seed customer data."""
    return ServiceContainer(
        customer_db=CustomerDatabase(CUSTOMERS),
        policy_engine=PolicyEngine(),
        financial_system=FinancialSystem(),
        escalation_queue=EscalationQueue(),
        audit_log=AuditLog(),
    )


from dotenv import find_dotenv, load_dotenv

# Load ANTHROPIC_API_KEY from .env (find_dotenv walks up from the notebooks/ dir)
load_dotenv(find_dotenv(), override=False)

client = anthropic.Anthropic()

# C003 scenario with PII: $600 refund + credit card number in message
# This single message exercises all 6 CCA patterns:
#   - $600 > $500 threshold → escalation (Pattern 1)
#   - Credit card number → PII redaction (Pattern 2)
#   - 5 focused tools used → tool design (Pattern 3)
#   - ContextSummary for session state → context management (Pattern 4)
#   - Prompt caching on system prompt → cost optimization (Pattern 5)
#   - EscalationRecord structured handoff → handoffs (Pattern 6)
user_message = (
    "Hi, I need a refund for my order. My customer ID is C003 and my order is O003. "
    "The item was defective — I paid $600 and want it back on my card ending in 1234. "
    "My card number is 4111-1111-1111-1234 in case you need it for the refund."
)

# Fresh services — shared across all 6 pattern sections
services = make_services()
callbacks = build_callbacks()

print("Scenario: C003 Carol Martinez — $600 refund with PII")
print(f"Message contains credit card: {'4111' in user_message}")
print("Amount: $600 (> $500 threshold — will escalate)")
print(f"\nMessage: {user_message[:100]}...")

## Pattern 1: Escalation (Deterministic Business Rules)

<div style="border-left: 4px solid #28a745; padding: 12px 16px; background: #f0fff4; margin: 8px 0;">
<strong>Correct Approach:</strong> The escalation rules live in <code>callbacks.py</code>, not in the prompt. For $600 the policy check returns <code>requires_review: true</code>, and on most runs Claude escalates on its own from that tool result. If it tries <code>process_refund</code> anyway, the callback blocks it and the loop forces <code>escalate_to_human</code> with <code>tool_choice</code>. If it ends its turn with the flag set and nothing queued, the loop forces escalation then too. Whichever path a run takes, the record lands in the escalation queue, and that queue is what we verify.
</div>

The run below uses the cached system prompt so that Pattern 5 has real cache numbers to show later.

In [ ]:
print("Running agent loop with deterministic callback enforcement...")
result = run_agent_loop(
    client,
    services,
    user_message,
    get_system_prompt_with_caching(),  # cached prompt: Pattern 5 reads the usage later
    callbacks=callbacks,
)

print(f"Stop reason: {result.stop_reason}")
print(f"Tool calls: {[tc['name'] for tc in result.tool_calls]}")

# Test the store: the escalation queue is the persistent state the rule protects.
escalations = services.escalation_queue.get_escalations()
print(f"\nEscalation queue: {len(escalations)} record(s)")

if result.stop_reason == "escalated":
    path = "forced by the loop with tool_choice (the backstop)"
elif escalations:
    path = "Claude called escalate_to_human on its own after check_policy flagged review"
else:
    path = "none"
print(f"Escalation path this run: {path}")

if escalations:
    print(f"Reason recorded: {escalations[-1].escalation_reason}")
    print("\nPattern 1 PASS: $600 request ended in the escalation queue, refund not processed")
else:
    print("\nPattern 1 FAIL: nothing queued — re-run this cell")
print(f"Refunds processed: {len(services.financial_system.get_processed())}")

> **CCA Exam Tip:** Deterministic business rules in code, not LLM confidence scores.
>
> - The rule `amount > 500` lives in a callback. It blocks `process_refund` and returns `action_required: escalate_to_human`; the loop then forces the call with `tool_choice`
> - Most runs never reach that path, because the policy check's tool result already tells Claude to escalate. The callback is the backstop that makes the outcome certain
> - Verify the store, not the transcript: the escalation queue has a record and the financial system has no refund
> - **Exam signal → Answer:** "escalate when uncertain" → WRONG → use amount thresholds and tier rules

## Pattern 2: Compliance (Programmatic PII Redaction)

<div style="border-left: 4px solid #28a745; padding: 12px 16px; background: #f0fff4; margin: 8px 0;">
<strong>Correct Approach:</strong> The <code>compliance_callback</code> in <code>callbacks.py</code> runs a regex over every <code>log_interaction</code> result and redacts card numbers before they reach the audit log. It runs whether or not Claude followed the prompt.
</div>

Be honest about what one live run can show. Claude usually does not copy the card number into a log entry, so on most runs the callback has nothing to redact and the audit log is clean for that reason. The cell reports which case happened. Notebook 02 has a deterministic replay that forces the card through and shows the redaction itself.

The cell also shows the callback's **coverage**: it protects the audit log only. The card in the customer's message is still in the raw message list, and that matters in Pattern 6.

In [ ]:
# Test the store: the audit log is the persistent state the callback protects.
log_entries = services.audit_log.get_entries()
print(f"Audit log entries: {len(log_entries)}")

raw_card = "4111-1111-1111-1234"
redacted_card = "****-****-****-1234"
pii_leaked = any(raw_card in entry.details for entry in log_entries)
pii_redacted = any(redacted_card in entry.details for entry in log_entries)

print("\nAudit log check:")
print(f"  Raw card number in log:   {pii_leaked}")
print(f"  Redacted pattern in log:  {pii_redacted}")
for entry in log_entries:
    print(f"  - {entry.action}: {entry.details[:120]}")

print()
if pii_leaked:
    print("Pattern 2 FAIL: raw card reached the audit log — check compliance_callback")
elif pii_redacted:
    print("Pattern 2 PASS: Claude wrote the card into a log entry and the callback redacted it")
else:
    print("Pattern 2 PASS (no redaction needed): Claude never wrote the card into a log entry,")
    print("  so the callback had nothing to do this run. See Notebook 02 for the forced case.")

# Coverage: the callback protects the audit log, nothing else.
raw_card_in_messages = raw_card in json.dumps(result.messages, default=str)
print(f"\nRaw card still in the message list: {raw_card_in_messages}")
print("  (the callback does not touch messages — remember this for the handoff in Pattern 6)")

> **CCA Exam Tip:** Programmatic redaction in callbacks, not prompt instructions.
>
> - `compliance_callback` runs regex on every `log_interaction` call — no prompt required
> - System prompt says "protect PII" but callbacks ENFORCE it — code beats text
> - A callback protects exactly the path it is registered on. The audit log is covered; the raw message list is not
> - **Exam signal → Answer:** "add PII rules to system prompt" → WRONG → use programmatic callbacks

<div style="border-left: 4px solid #2196F3; padding: 10px; margin: 10px 0; background-color: #E3F2FD;">
<strong>CCA Meta-Pattern:</strong> This project's own pre-commit hooks (<code>.pre-commit-config.yaml</code>) follow the same principle — programmatic enforcement (ruff, nbstripout) rather than relying on developers to remember code style rules. Just as <code>compliance_callback</code> runs on every <code>log_interaction</code> call without a prompt, <code>ruff</code> runs on every commit without a reminder.
</div>

## Pattern 3: Tool Design (5 Focused Tools)

<div style="border-left: 4px solid #28a745; padding: 12px 16px; background: #f0fff4; margin: 8px 0;">
<strong>Correct Approach:</strong> 4-5 focused tools per agent, each with a single clear purpose. The Swiss Army anti-pattern uses 15+ tools — Claude spends tokens reasoning about which of 15 tools to call instead of solving the problem.
</div>

In [ ]:
from customer_service.tools.definitions import TOOLS

print(f"Correct pattern tool count: {len(TOOLS)}")
print("Tools used:")
for tool in TOOLS:
    print(f"  - {tool['name']}: {tool['description'][:60]}...")

print()
unique_tools_called = list({tc["name"] for tc in result.tool_calls})
print(f"Tools called in this scenario: {unique_tools_called}")
print(f"Unique tools used: {len(unique_tools_called)}")

# Verify tool count constraint
assert len(TOOLS) <= 5, f"Tool count violation: {len(TOOLS)} > 5"
print(f"\nPattern 3 PASS: {len(TOOLS)} focused tools (CCA rule: 4-5 max per agent)")

> **CCA Exam Tip:** 4-5 focused tools per agent; use coordinator-subagent pattern for more.
>
> - Each tool does ONE thing: `lookup_customer`, `check_policy`, `process_refund`, `escalate_to_human`, `log_interaction`
> - More tools = more reasoning overhead + higher error rate
> - **Exam signal → Answer:** "add more tools to handle more cases" → WRONG → use coordinator-subagent pattern

<div style="border-left: 4px solid #2196F3; padding: 10px; margin: 10px 0; background-color: #E3F2FD;">
<strong>CCA Meta-Pattern:</strong> The project's CI pipeline (<code>.github/workflows/ci.yml</code>) uses <code>--allowedTools</code> to sandbox the Claude reviewer to <code>Read</code>, <code>Grep</code>, and <code>Glob</code> only — the same 'focused tools' principle you just saw with 5 tools vs 15. A CI agent that can only read files cannot accidentally modify the codebase, just as a customer lookup tool that does NOT modify customer data cannot corrupt records.
</div>

## Pattern 4: Context Management (Structured Summaries)

<div style="border-left: 4px solid #28a745; padding: 12px 16px; background: #f0fff4; margin: 8px 0;">
<strong>Correct Approach:</strong> <code>ContextSummary</code> keeps a fixed-field schema — customer_id, issue_type, tools_called, decisions_made, pending_actions — and its rendered size stays under <code>TOKEN_BUDGET</code> (about 300 estimated tokens, 1,200 characters). A raw transcript grows with every message.
</div>

Here the summary is filled from the tool calls the run actually made. Notebook 05 sends the summary back to the model on every turn and measures the growth; this cell only shows the size difference for the one run above.

In [ ]:
def describe_tool_call(call: dict) -> str:
    """One short line per tool call, built from name and inputs (never the raw result)."""
    args = ", ".join(f"{k}={v}" for k, v in call["input"].items() if k != "customer_id")
    return f"{call['name']}({args})"[:80]


summary = ContextSummary()
summary.customer_id = "C003"
summary.issue_type = "defective_item_refund_escalated"
for tc in result.tool_calls:
    summary.update(tc["name"], describe_tool_call(tc))

print(f"TOKEN_BUDGET: {TOKEN_BUDGET} estimated tokens (~{TOKEN_BUDGET * 4} chars)")
print(f"Structured summary token_estimate: {summary.token_estimate}")
print(f"Under budget: {summary.token_estimate <= TOKEN_BUDGET}")
print()
print("Structured context output:")
print(summary.to_system_context())

raw_context_len = len(json.dumps(result.messages, default=str))
structured_context_len = len(summary.to_system_context())
print("\nSize comparison for this one run:")
print(f"  Raw messages dump:  {raw_context_len:,} chars")
print(f"  Structured summary: {structured_context_len:,} chars")
print(f"  Ratio: {raw_context_len / structured_context_len:.1f}x smaller with structured")
print("\nPattern 4 PASS: summary under budget" if summary.token_estimate <= TOKEN_BUDGET
      else "\nPattern 4 FAIL: summary over budget")

> **CCA Exam Tip:** Structured JSON summaries, not raw transcripts.
>
> - Raw transcripts grow O(n) — each turn adds full message + response + tool results
> - `ContextSummary` fixed-field schema: token_estimate never exceeds TOKEN_BUDGET
> - Early facts (customer_id, issue_type) survive every compaction — they're in named fields
> - **Exam signal → Answer:** "quality drops in longer sessions" → lost-in-middle → use ContextSummary

## Pattern 5: Cost Optimization (Prompt Caching)

<div style="border-left: 4px solid #28a745; padding: 12px 16px; background: #f0fff4; margin: 8px 0;">
<strong>Correct Approach:</strong> <code>get_system_prompt_with_caching()</code> marks the large POLICY_DOCUMENT block with <code>cache_control</code>. The first API call writes it to cache at 1.25x; every later call reads it at 10%. The anti-pattern is the Batch API — a 50% discount bought with up to 24-hour latency, which is <strong>completely wrong</strong> for live customer support.
</div>

The run in Pattern 1 used the cached prompt and made several API calls (one per tool round-trip). So its usage should show one cache write and reads on the remaining calls. The cost line compares what the run cost against what the same tokens would have cost with no caching.

In [ ]:
usage = result.usage
print("Token usage from the Pattern 1 run (all API calls in the loop):")
print(f"  Input tokens:       {usage.input_tokens:,}")
print(f"  Output tokens:      {usage.output_tokens:,}")
print(f"  Cache write tokens: {usage.cache_creation_input_tokens:,}")
print(f"  Cache read tokens:  {usage.cache_read_input_tokens:,}")

cached_prompt = get_system_prompt_with_caching()
print(f"\nget_system_prompt_with_caching() returns {len(cached_prompt)} blocks:")
for idx, block in enumerate(cached_prompt):
    marker = block.get("cache_control", "none")
    print(f"  Block {idx}: {len(block['text']) // 4:,} tokens (estimate), cache_control={marker}")

# What the same run would have cost with every token billed at the input rate
from types import SimpleNamespace

uncached_input = usage.input_tokens + usage.cache_read_input_tokens + usage.cache_creation_input_tokens
uncached_cost = estimate_cost(SimpleNamespace(input_tokens=uncached_input, output_tokens=usage.output_tokens))
actual_cost = estimate_cost(usage)
print(f"\nEstimated cost, this run:      ${actual_cost:.6f}")
print(f"Estimated cost, no caching:    ${uncached_cost:.6f}")
print(f"Saving: {(1 - actual_cost / uncached_cost) * 100:.1f}%")

print()
if usage.cache_creation_input_tokens > 0 or usage.cache_read_input_tokens > 0:
    print("Pattern 5 PASS: policy block cached — write on the first call, reads after")
else:
    print("Pattern 5 FAIL: no cache activity — check that the run used get_system_prompt_with_caching()")

> **CCA Exam Tip:** Prompt caching (90% savings on reads) for repeated context.
>
> - `get_system_prompt_with_caching()` marks the large POLICY_DOCUMENT block with `cache_control`
> - Cache reads cost 10% of normal input tokens; the write costs 125% once. The blended saving depends on how much of each call is the static block
> - **NEVER use Batch API for live customer support** — 24-hour latency is incompatible with real-time service
> - **Exam signal → Answer:** "reduce costs for live chat" → prompt caching → NOT Batch API

## Pattern 6: Structured Handoffs (Schema-Enforced Record)

<div style="border-left: 4px solid #28a745; padding: 12px 16px; background: #f0fff4; margin: 8px 0;">
<strong>Correct Approach:</strong> The <code>EscalationRecord</code> in the escalation queue is the handoff payload — 8 named fields, validated against the tool's input schema before the handler runs, so it has the same shape whether Claude escalated on its own or was forced with <code>tool_choice</code>. The anti-pattern dumps the full <code>messages</code> list as raw JSON, including API-internal <code>tool_use</code> blocks and, in this scenario, the customer's raw card number.
</div>

In [ ]:
escalations = services.escalation_queue.get_escalations()

if escalations:
    escalation_dict = escalations[-1].model_dump()
    structured_output = json.dumps(escalation_dict, indent=2, default=str)
    structured_len = len(structured_output)
    print("STRUCTURED HANDOFF (EscalationRecord):")
    print(structured_output)

    raw_dump = format_raw_handoff(result.messages)
    raw_len = len(raw_dump)

    print("\nHandoff comparison:")
    print(f"  Structured EscalationRecord: {structured_len:,} chars, {len(escalation_dict)} named fields")
    print(f"  Raw conversation dump:       {raw_len:,} chars, {len(result.messages)} messages")
    print(f"  Ratio: raw is {raw_len / structured_len:.1f}x larger")
    print(f"  Raw contains tool_use blocks:   {'tool_use' in raw_dump}")
    print(f"  Record contains tool_use blocks: {'tool_use' in structured_output}")

    # Pattern 2's coverage gap shows up here: the raw dump carries the customer's card number
    # to the human agent. The record only carries it if Claude copied it into a field.
    print(f"  Raw contains the raw card number:    {raw_card in raw_dump}")
    print(f"  Record contains the raw card number: {raw_card in structured_output}")
    if raw_card in structured_output:
        print("  NOTE: Claude copied the card into the record. compliance_callback is registered on")
        print("        log_interaction only; a real deployment registers it on escalate_to_human too.")
    print("\nPattern 6 PASS: structured EscalationRecord in escalation_queue, no API noise")
else:
    print("No escalation found — rerun cells from Pattern 1 section")

> **CCA Exam Tip:** Structured JSON handoffs from a schema-enforced tool call, not raw conversation dumps.
>
> - `escalate_to_human` has a fixed input schema, so every record has the same fields; `tool_choice={"type": "tool", "name": "escalate_to_human"}` is the backstop that guarantees the call happens
> - `EscalationRecord` has 8 fields: customer_id, customer_tier, issue_type, disputed_amount, escalation_reason, recommended_action, conversation_summary, turns_elapsed
> - Raw dumps include `tool_use` blocks with internal tool IDs — useless noise for human agents — and whatever PII the customer typed
> - **Exam signal → Answer:** "human agent can't find relevant info" → raw handoff → use EscalationRecord

<div style="border-left: 4px solid #2196F3; padding: 10px; margin: 10px 0; background-color: #E3F2FD;">
<strong>CCA Meta-Pattern:</strong> For a hands-on walkthrough of how this project uses CCA patterns in its own infrastructure — CI pipeline flags, CLAUDE.md hierarchy, custom skills, and pre-commit hooks — see <strong>Notebook 08: Meta-Teaching</strong> (<code>notebooks/08_meta_teaching.ipynb</code>). The same principles that govern the agent code govern the project toolchain.
</div>

## Extension Exercises (TODOs)

In [ ]:
# TODO: Add a new escalation rule — escalate when refund_count > 3 in 30 days
# HINTS:
#   1. Add "refund_count" tracking to context dict
#   2. Check context.get("refund_count", 0) > 3 in a custom callback
#   3. Return CallbackResult(action="block", ...) with escalation message
# EXPECTED: A customer with 4+ prior refunds triggers escalation even for $50
try:
    raise NotImplementedError("TODO: implement frequency-based escalation rule")
except NotImplementedError:
    print("TODO not yet implemented - skipping extension. Core patterns demonstrated above.")

In [ ]:
# TODO: Add a premium-tier customer with a partial refund scenario
# HINTS:
#   1. Create CustomerProfile(
#          customer_id="C007", name="Student Test", tier=CustomerTier.PREMIUM, ...
#      )
#   2. Add to services.customer_db manually or use a local dict
#   3. Run the agent loop with C007 — PREMIUM tier limit is higher than REGULAR
# EXPECTED: $400 refund should NOT require review for PREMIUM tier
student_customer = None  # Replace with: CustomerProfile(...)
if student_customer is None:
    print("Using default C003 scenario (implement TODO to use custom customer)")
else:
    print(f"Custom customer: {student_customer.customer_id} ({student_customer.tier})")

## Summary: All 6 CCA Patterns Demonstrated

| Pattern | Correct Approach | Key Verification (the store, not the transcript) |
|---------|-----------------|-----------------|
| 1. Escalation | Deterministic callback rules | Record in `escalation_queue`, nothing in `financial_system` |
| 2. Compliance | Programmatic PII redaction | No raw card in `audit_log`; coverage is the log only |
| 3. Tool Design | 5 focused tools | `len(TOOLS) <= 5` |
| 4. Context | Structured summaries | `token_estimate <= TOKEN_BUDGET` |
| 5. Cost | Prompt caching | `cache_creation_input_tokens` or `cache_read_input_tokens > 0` |
| 6. Handoffs | Schema-enforced EscalationRecord | 8 named fields, no `tool_use` noise |

**Core CCA principle:** Programmatic enforcement beats prompt-based guidance.
Code (callbacks, schemas, tool_choice, regex) enforces rules; system prompts provide context.